## Session 3: Grouping, Aggregations & Feature Engineering

**Date:** 18/05/2026  
**Objective:** Apply grouping and aggregation techniques to uncover patterns in gaming communities. Create new features (engagement ratio, tier scores, age groups, etc.) to enrich the dataset for deeper analysis. Prepare the data for visualisation and statistical testing.

---

### 3.1 Agenda for Session 3

| Step | Activity | Tools | Expected Output |
|------|----------|-------|------------------|
| 1 | Load the cleaned dataset | `pd.read_csv()` | DataFrame (357 rows × 21 columns) |
| 2 | Basic groupby – `genre` and mean `member_count` | `.groupby()`, `.mean()` | Genre size ranking |
| 3 | Multiple aggregations – `.agg()` with custom functions | `.agg()` | Summary stats per group |
| 4 | Multi‑level grouping – `genre` + `community_tier` | `.groupby()` | Counts and averages per combination |
| 5 | Ranking within groups – largest community per region | `.rank()` or `.idxmax()` | Ranked communities |
| 6 | Filtering groups – keep only genres with >10 communities | `.filter()` | Filtered DataFrame |
| 7 | Transform – add group means as new column | `.transform()` | New column with genre averages |
| 8 | Create `engagement_ratio` = `active_players / member_count` | Column arithmetic | New feature |
| 9 | Create `tier_score` mapping `Amateur=1`, `Semi-Pro=2`, `Professional=3`, `Elite=4` | `.map()` or `.replace()` | Numeric tier column |
| 10 | Create `age_group` using `pd.cut()` | `pd.cut()` | Categorical age groups |
| 11 | Create `community_age` = 2025 – `year_founded` | Column arithmetic | Community age in years |
| 12 | Create `has_coaching` boolean column | `>` operator | Binary flag |
| 13 | Pivot table – `genre` × `community_tier` for mean `member_count` | `pd.pivot_table()` | 2D summary table |
| 14 | Crosstab – `genre` vs `language` (normalised) | `pd.crosstab(normalize='rows')` | Proportions table |
| 15 | Cumulative sum of `member_count` per genre over `year_founded` | `.groupby().cumsum()` | Running total column |
| 16 | Save enhanced DataFrame with new features | `.to_csv()` | `gaming_communities_enhanced.csv` |

---

### 3.2 Initial Observations (Before Starting)

- The cleaned dataset has **357 rows**, **21 columns**, **0 missing values**.
- Key numeric columns: `member_count`, `active_players`, `top_player_rank`, `social_media_platforms`, `discord_server_size`, `reddit_community_size`, `avg_member_age`, `coaching_staff`, `content_creators`, `tournaments_hosted`.
- Key categorical columns: `genre`, `community_tier`, `primary_platform`, `region`, `language`, `country_of_origin`.
- No revenue or win rate columns (dropped in Session 2) – we will use other proxies for success (e.g., `top_player_rank`, `tournaments_hosted`).
- Some groups (e.g., `Strategy` genre) have very few communities – may need filtering.

---

### 3.3 Deliverables by End of Session

- [ ] Jupyter notebook with all grouping, aggregation, and feature engineering steps.
- [ ] New features added to the DataFrame.
- [ ] Enhanced dataset saved as `gaming_communities_enhanced.csv`.
- [ ] Summary of key findings (e.g., which genre has highest average engagement, which tier is most active).
- [ ] At least one pivot table or crosstab with meaningful insight.

---

### 3.4 Next Steps (Preview)

**Session 4:** Visualisation & Statistical Analysis  
- Create static (Matplotlib/Seaborn) and interactive (Plotly) charts.
- Perform hypothesis tests (t‑tests, ANOVA, chi‑square) on groups.
- Build a small dashboard to present key metrics.

---

### 3.5 Tools & Libraries

- Python 3.10+
- pandas, numpy
- (matplotlib, seaborn, plotly for visualisation – will be used in Session 4)

---

*This notebook builds on the cleaned data to extract patterns and create features for advanced analysis.*

In [54]:
#importing essential libraries
import pandas as pd
import numpy as np

In [55]:
#loading the dataset
gaming_data = pd.read_csv('gaming_communities_data_cleaned.csv')

In [57]:
genre = gaming_data.groupby(['genre'])
print('Genre with the highest average community size:')
genre['member_count'].mean().nlargest(1)

Genre with the highest average community size:


genre
Fighting    5596.4375
Name: member_count, dtype: float64

Genre with the highest average community size is Fighting (5596.437), followed by MOBA (3551.31), FPS (3242.57) and so on 

In [59]:
community_tier = gaming_data.groupby(['community_tier'])
community_tier['tournaments_hosted'].sum()

community_tier
Amateur          20
Elite            21
Professional    123
Semi-Pro        525
Name: tournaments_hosted, dtype: int64

Semi‑Pro communities host the most tournaments (525), followed by Professional (123). Amateur and Elite host very few.

In [61]:
prim_platform = gaming_data.groupby(['primary_platform'])
print('Platform with the highest average active players:')
prim_platform['active_players'].median().nlargest(1)

Platform with the highest average active players:


primary_platform
Twitter/X    311.0
Name: active_players, dtype: float64

Platform with the highest average of active players is Twitter/X (311.0), followed by Twitch (303.0), YouTube (254.5) and so on

In [46]:
region = gaming_data.groupby(['region'])
region['avg_member_age'].agg(['min', 'max'])

,min,max
region,,
AF,19.1,31.8
AS,15.7,29.9
EU,14.0,37.9
OC,17.1,34.6
SA,14.7,32.6
Unknown,14.0,38.3


The region with the youngest member average age is EU (europe) with minimum avg age of 14.0 and maximum avg age of 37.9

In [53]:
language = gaming_data.groupby(['language'])
language['community_name'].count().sort_values(ascending=False)

language
English       141
Spanish        62
Portuguese     32
Chinese        28
Korean         24
Russian        19
Arabic         14
French         13
German         13
Japanese       11
Name: community_name, dtype: int64

The top three most common languages used by communities are English (141), Spanish (62) and Portuguese (32)

In [62]:
gaming_data.groupby(['genre']).agg({
    'member_count': 'mean',
    'tournaments_hosted': 'sum',
    'top_player_rank': 'median'
})

,member_count,tournaments_hosted,top_player_rank
genre,,,
Battle Royale,3241.642857,103,3505.5
FPS,3242.579618,361,5025.0
Fighting,5596.437500,20,4581.0
MMORPG,2427.523810,4,4276.0
MOBA,3551.312500,150,3428.0
Sports,3059.240000,51,4693.0
Strategy,346.750000,0,3634.5


In [67]:
gaming_data.groupby(['country_of_origin'])['member_count'].sum().sort_values(ascending=False).iloc[:5]

country_of_origin
United States    202982
Taiwan           174737
Mexico           146408
China            104804
South Africa      74034
Name: member_count, dtype: int64

In [69]:
gaming_data.groupby(['community_tier'])['member_count'].std().sort_values(ascending=False)

community_tier
Professional    18501.523943
Semi-Pro         3476.735288
Amateur           498.502645
Elite               0.000000
Name: member_count, dtype: float64

In [70]:
gaming_data.groupby(['genre'])['primary_game'].nunique().sort_values(ascending=False)

genre
FPS              5
Battle Royale    3
MOBA             3
Fighting         2
MMORPG           2
Sports           1
Strategy         1
Name: primary_game, dtype: int64

In [71]:
gaming_data['discord_to_reddit_ratio'] = gaming_data['discord_server_size'] / gaming_data['reddit_community_size']

gaming_data['discord_to_reddit_ratio'] = gaming_data['discord_to_reddit_ratio'].replace([np.inf, -np.inf], 0).fillna(0)

gaming_data.groupby(['region'])['discord_to_reddit_ratio'].mean()

region
AF         1.701499
AS         1.833267
EU         2.299356
OC         3.243439
SA         2.069069
Unknown    2.317666
Name: discord_to_reddit_ratio, dtype: float64

In [74]:
gaming_data.groupby(['genre', 'community_tier']).size().reset_index(name='count')

,genre,community_tier,count
0,Battle Royale,Amateur,40
1,Battle Royale,Professional,6
2,Battle Royale,Semi-Pro,24
3,FPS,Amateur,83
4,FPS,Elite,3
5,FPS,Professional,12
6,FPS,Semi-Pro,59
7,Fighting,Amateur,8
8,Fighting,Professional,1
9,Fighting,Semi-Pro,7


In [76]:
gaming_data.groupby(['region', 'primary_platform'])['active_players'].mean().reset_index(name='Average')

,region,primary_platform,Average
0,AF,Discord,1592.090909
1,AF,Reddit,587.166667
2,AF,Twitch,1059.000000
3,AF,YouTube,1133.750000
4,AS,Discord,1215.363636
5,AS,Reddit,899.857143
6,AS,Twitch,3695.571429
7,AS,Twitter/X,269.142857
8,AS,YouTube,2290.666667
9,EU,Discord,534.639344
